# Predicting gene expression from H&E

SIGMMA's image encoder was aligned with spatial transcriptomics (ST) during training, so its
H&E embeddings should carry information about gene expression. A simple way to measure this
is a **linear probe**: fit a ridge regression from tile embeddings to the measured expression
of the same tiles, then predict expression on a different section from H&E alone.

In this tutorial we:

1. embed every tissue tile of two HEST-1k lung adenocarcinoma sections **from H&E only**,
   with SIGMMA and with the UNI2-h backbone it was built on,
2. compute each tile's measured expression for a fixed gene panel,
3. train a ridge regressor on `TENX118` and predict expression on `TENX141`, and
4. compare per-gene Pearson correlations and look at predicted expression maps.

**Requirements:** a GPU, the UNI2-h weights and a SIGMMA checkpoint (see {doc}`../installation`).
The probe follows the HEST-1k benchmark recipe: standardise, reduce to 256 principal
components, ridge regression.

## Paths

In [ ]:
from pathlib import Path

DATA_DIR = Path("./data/hest_1k_lung")
TRAIN_SECTION = "TENX118"   # fit the probe here
TEST_SECTION = "TENX141"    # predict expression here from H&E only
CKPT = Path("./weights/sigmma.pth")
UNI_DIR = Path("./weights/UNI2-h")

def section_files(name):
    d = DATA_DIR / name
    return dict(image_file=d / "hematoxylin_eosin.zarr",
                indices_file=d / "index_mask_HAE_stride_256_tile_256.zarr",
                spatial_file=d / "spatial_data_processed.h5")

FOV = 256                  # tile size in H&E pixels
UM_PER_PX = 0.5            # H&E resolution
TISSUE_THRESHOLD = 20000   # SpatialPatchDataset's default tissue filter
N_GENES = 50               # size of the predicted gene panel

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import zarr
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from ml_collections.config_dict import ConfigDict
from scipy.stats import pearsonr
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge

from sigmma.utils import load_model
from sigmma.model import UNI

plt.rcParams["figure.dpi"] = 80
device = torch.device("cuda")

## Load the models

The SIGMMA checkpoint and configuration are the same as in {doc}`multiscale_embeddings`.
As a baseline we also load the UNI2-h backbone on its own, without SIGMMA's LoRA adapters
and projection heads.

In [ ]:
config = ConfigDict({
    "model": {
        "name": "sigmma",
        "checkpoint_path": str(CKPT),
        "pretrained_image_encoder": "UNI",
        "pretrained_image_encoder_ckpt_path": str(UNI_DIR),
        "image_encoder_input_dim": 1536,
        "image_encoder_output_dim": 512,
        "spatial_encoder_input_dim": 384,
        "spatial_encoder_output_dim": 512,
        "num_resolutions": 3,
        "contrastive_temperature": 0.07,
        "gumbel_temperature": 0.5,
        "edge_threshold": 0.5,
        "candidate_top_k": 8,
        "candidate_sim_threshold": 0.5,
        "dropout": 0.1,
        "activation": "leaky_relu",
        "aggregate": "attn_pool",
        "num_graph_layers": 2,
        "image_mlp_hidden_dim": 2048,
        "image_attn_pool_heads": 8,
        "lora": {
            "enabled": True, "r": 16, "alpha": 32, "dropout": 0.05,
            "target_modules": ["qkv", "proj"], "bias": "none", "per_scale": False,
        },
    },
})

model = load_model(
    config,
    pretrained_image_encoder=config.model.pretrained_image_encoder,
    pretrained_ckpt_path=config.model.checkpoint_path,
    train=False,
)
model.to(device).eval()

uni = UNI(checkpoint_path=str(UNI_DIR)).to(device).eval()
print("UNI2-h feature dim:", uni.feature_dim)

## Tissue tiles

Each section comes with a tile index: the top-left corner (`y_start`, `x_start`) of every
256 × 256 px tile (128 × 128 µm at 0.5 µm/px) and a tissue score. We keep the tiles whose
score passes the threshold that `SpatialPatchDataset` uses by default.

In [ ]:
def load_tiles(name):
    idx = np.asarray(zarr.open(str(section_files(name)["indices_file"]), mode="r")[:])
    idx = idx[idx[:, 2] >= TISSUE_THRESHOLD]
    return pd.DataFrame({"y_start": idx[:, 0], "x_start": idx[:, 1]})

tiles = {name: load_tiles(name) for name in (TRAIN_SECTION, TEST_SECTION)}
for name, t in tiles.items():
    print(f"{name}: {len(t):,} tissue tiles")

## Embed tiles from H&E only

`model.get_image_features` runs only SIGMMA's image branch and returns the L2-normalised
**macro**-scale embedding (512-d), which covers the whole tile. No transcriptomics is used.
For UNI2-h we take the 1536-d class token after the standard UNI preprocessing: resizing
to 224 px and ImageNet normalisation.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

class TileDataset(Dataset):
    def __init__(self, name, tiles):
        self.he = zarr.open(str(section_files(name)["image_file"]), mode="r")
        self.xy = tiles[["x_start", "y_start"]].to_numpy()

    def __len__(self):
        return len(self.xy)

    def __getitem__(self, i):
        x, y = self.xy[i]
        return torch.from_numpy(np.asarray(self.he[:, y:y + FOV, x:x + FOV])).float() / 255.0

@torch.no_grad()
def embed(name, batch_size=64):
    dl = DataLoader(TileDataset(name, tiles[name]), batch_size=batch_size, num_workers=4)
    z_sigmma, z_uni = [], []
    for img in dl:
        img = img.to(device)
        z_sigmma.append(model.get_image_features(img).float().cpu())
        x = F.interpolate(img, size=(224, 224), mode="bilinear", align_corners=False)
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        with torch.autocast("cuda", dtype=torch.bfloat16):
            z_uni.append(uni.model(x).float().cpu())
    return {"SIGMMA": torch.cat(z_sigmma).numpy(), "UNI2-h": torch.cat(z_uni).numpy()}

emb = {name: embed(name) for name in (TRAIN_SECTION, TEST_SECTION)}
for name, e in emb.items():
    print(name, {k: v.shape for k, v in e.items()})

## Measured expression per tile

`spatial_data_processed.h5` stores log-normalised expression (`log1p`) for every cell and the
cell centroids in µm in `obsm["spatial"]`. The target for a tile is the mean expression over
the cells whose centroid falls inside it (a pseudobulk). Tiles lie on a 256 px grid, so each
cell is assigned to a tile by integer division of its pixel coordinates.

In [ ]:
def tile_expression(name):
    adata = sc.read_h5ad(section_files(name)["spatial_file"])
    t = tiles[name]
    assert (t.x_start % FOV == 0).all() and (t.y_start % FOV == 0).all()
    px = np.asarray(adata.obsm["spatial"]) / UM_PER_PX
    cell_key = pd.MultiIndex.from_arrays([(px[:, 0] // FOV).astype(int), (px[:, 1] // FOV).astype(int)])
    tile_key = pd.MultiIndex.from_arrays([t.x_start // FOV, t.y_start // FOV])
    tile_of_cell = tile_key.get_indexer(cell_key)          # -1 for cells outside kept tiles
    inside = tile_of_cell >= 0
    A = sp.csr_matrix((np.ones(inside.sum()), (tile_of_cell[inside], np.flatnonzero(inside))),
                      shape=(len(t), adata.n_obs))
    n_cells = np.asarray(A.sum(axis=1)).ravel()
    X = sp.csr_matrix(adata.X)
    Y = np.asarray((sp.diags(1 / np.maximum(n_cells, 1)) @ A @ X).todense())
    return pd.DataFrame(Y, columns=adata.var_names), n_cells

expr, n_cells = {}, {}
for name in (TRAIN_SECTION, TEST_SECTION):
    expr[name], n_cells[name] = tile_expression(name)
    print(f"{name}: median {np.median(n_cells[name]):.0f} cells per tile, "
          f"{(n_cells[name] == 0).sum()} tiles without cells")

The two sections were profiled with different Xenium panels, so we can only predict genes
measured in both. Among those we predict the 50 genes that vary most across the training
tiles. Tiles without cells have no target and are dropped.

In [ ]:
shared = expr[TRAIN_SECTION].columns.intersection(expr[TEST_SECTION].columns)
panel = expr[TRAIN_SECTION][shared].var().sort_values(ascending=False).index[:N_GENES].tolist()
print(f"{len(shared)} shared genes; predicting {len(panel)}:")
print(", ".join(panel))

keep = {name: n_cells[name] > 0 for name in (TRAIN_SECTION, TEST_SECTION)}
Y_train = expr[TRAIN_SECTION].loc[keep[TRAIN_SECTION], panel].to_numpy()
Y_test = expr[TEST_SECTION].loc[keep[TEST_SECTION], panel].to_numpy()
print("train", Y_train.shape, "test", Y_test.shape)

## Train on TENX118, predict TENX141

For each embedding we fit the probe on all training tiles and predict the test section. The
ridge penalty follows the HEST-1k benchmark (and `sigmma.evaluation.linear_probing`).
Performance is the Pearson correlation between measured and predicted expression across
test tiles, computed for each gene.

In [ ]:
def linear_probe(Z_train, Y_train, Z_test, n_components=256, seed=0):
    n_components = min(n_components, Z_train.shape[1])
    pipe = make_pipeline(
        StandardScaler(),
        PCA(n_components=n_components, random_state=seed),
        Ridge(alpha=100 / (n_components * Y_train.shape[1]), solver="lsqr", random_state=seed),
    )
    pipe.fit(Z_train, Y_train)
    return pipe.predict(Z_test)

pred, per_gene = {}, {}
for enc in ("SIGMMA", "UNI2-h"):
    Z_train = emb[TRAIN_SECTION][enc][keep[TRAIN_SECTION]]
    Z_test = emb[TEST_SECTION][enc][keep[TEST_SECTION]]
    pred[enc] = linear_probe(Z_train, Y_train, Z_test)
    per_gene[enc] = [pearsonr(Y_test[:, g], pred[enc][:, g])[0] for g in range(len(panel))]

per_gene = pd.DataFrame(per_gene, index=panel)
per_gene.agg(["mean", "median"]).round(3)

## Per-gene comparison

Each point is one gene; points above the diagonal are predicted better from SIGMMA embeddings
than from UNI2-h embeddings.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(per_gene["UNI2-h"], per_gene["SIGMMA"], s=18)
lo, hi = per_gene.min().min() - 0.05, per_gene.max().max() + 0.05
ax.plot([lo, hi], [lo, hi], color="grey", lw=1, ls="--")
for g, r in per_gene.assign(diff=per_gene["SIGMMA"] - per_gene["UNI2-h"]).nlargest(5, "SIGMMA").iterrows():
    ax.annotate(g, (r["UNI2-h"], r["SIGMMA"]), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel("Pearson r, UNI2-h"); ax.set_ylabel("Pearson r, SIGMMA")
ax.set_title(f"{len(panel)} genes, {TEST_SECTION} (held out)")
plt.tight_layout(); plt.show()

print(f"SIGMMA better for {(per_gene['SIGMMA'] > per_gene['UNI2-h']).sum()} of {len(panel)} genes")
per_gene.sort_values("SIGMMA", ascending=False).head(10).round(3)

## Predicted expression maps

Measured and predicted expression on the held-out section for the three genes SIGMMA
predicts best. Each pixel is one 128 × 128 µm tile; the colour scale is shared within a row.

In [ ]:
t_test = tiles[TEST_SECTION][keep[TEST_SECTION]]
rows, cols = (t_test.y_start // FOV).to_numpy(), (t_test.x_start // FOV).to_numpy()

def to_grid(values):
    grid = np.full((rows.max() + 1, cols.max() + 1), np.nan)
    grid[rows, cols] = values
    return grid

genes = per_gene["SIGMMA"].nlargest(3).index
fig, axes = plt.subplots(len(genes), 3, figsize=(15, 3.2 * len(genes)))
for i, gene in enumerate(genes):
    g = panel.index(gene)
    maps = {"measured": Y_test[:, g], "SIGMMA": pred["SIGMMA"][:, g], "UNI2-h": pred["UNI2-h"][:, g]}
    vmin, vmax = np.percentile(Y_test[:, g], [1, 99])
    for ax, (label, v) in zip(axes[i], maps.items()):
        im = ax.imshow(to_grid(v), cmap="viridis", vmin=vmin, vmax=vmax, interpolation="nearest")
        r = "" if label == "measured" else f"  (r = {per_gene.loc[gene, label]:.2f})"
        ax.set_title(f"{gene}: {label}{r}", fontsize=10); ax.axis("off")
    fig.colorbar(im, ax=axes[i].tolist(), shrink=0.8, label="mean log1p expression")
plt.show()

## Notes

- A linear probe measures how linearly accessible expression is in an embedding; it is not a
  tuned expression predictor. Non-linear regressors or fine-tuning usually score higher.
- Results depend on the gene panel. Here it is the 50 most variable genes shared by the two
  sections; change `N_GENES` or build `panel` from your own gene list.
- To compare more encoders across many sections, use `scripts/benchmark_gex_prediction.py`
  from a source checkout with the `benchmarks/` project installed.